# 🎨 [컴비 2조] 가짜 생성 B (제품 영역 인페인팅) – Colab 원클릭 노트북

### 🧭 한눈에 보는 실행 순서

#### A. 처음 생성 / 부족분 재뽑기 (GPU 런타임)
1. **Cell 1** (환경 준비 & 드라이브 마운트)
2. **Cell 2** (경로 자동 탐지 & 360장 스캔)
3. **Cell 3** (모델 로드) → **Cell 4** (엔진 로드)
4. **Cell 6** (`run_batch()` — 이미 있는 사진은 스킵하고 **부족분만 자동 생성**)

#### B. 검수 & 재뽑기 루프 (CPU 런타임도 OK!)
1. **Cell 7**: 새로 뽑힌 사진만 `fake/B/_재검수/` 물건별 폴더로 모으기
2. 👉 **구글 드라이브**에서 `fake/B/_재검수/` 열어서 망한 사진 삭제
3. **Cell 8**: `_재검수/` 폴더에서 사진 전부 `fake/B/`로 원복 + 빈 폴더 삭제
4. **Cell 9**: `sync_and_verify()` — 마스크·CSV 1:1 정리 + 부족분 확인
5. **Cell 10**: 현재 통과본 목록 저장 (`_approved_B.txt`) → 부족분 있으면 **Cell 6**으로 돌아가서 재뽑기!

> 💡 **팀 규약 (`docs/data_contract.md`) 100% 준수**:
> - 이미지: `data/fake/B/fakeB_{원본stem}_s{seed}.png` (24종 × 15장 = 360장)
> - 마스크: `data/fake/masks/fakeB_{원본stem}_s{seed}_mask.png`
> - 대응표: `data/fake/fake_mapping_B.csv`


## 1️⃣ [Cell 1] 환경 준비 (GPU 확인 + 라이브러리 설치 + 드라이브 마운트)


In [ ]:
# ==============================================================================
# 📌 [Cell 1] GPU 확인, 라이브러리 설치 & 구글 드라이브 마운트
# ==============================================================================
!nvidia-smi
!pip install -q -U diffusers transformers accelerate safetensors

from google.colab import drive
drive.mount("/content/drive")
print("✅ 환경 준비 및 드라이브 마운트 완료!")

## 2️⃣ [Cell 2] 경로 자동 탐지 & 360장 대상 스캔
> 새 계정으로 넘어가거나 맥(NFD) 한글 자소 분리가 되어 있어도 자동으로 `[공유폴더]`를 찾습니다.


In [ ]:
# ==============================================================================
# 📌 [Cell 2] 경로 설정 & 진짜 사진 자동 스캔 (B 대상 360장 자동 선별)
# ==============================================================================
import os, difflib
from collections import defaultdict

# 👉 공유 드라이브 경로 자동 탐지 (계정 변경 / 맥 NFD 자소분리 자동 대응!)
import unicodedata

def find_data_root(base="/content/drive/MyDrive"):
    candidates = [
        "/content/drive/MyDrive/[컴비] 팀프로젝트/[공유폴더]",
        unicodedata.normalize("NFD", "/content/drive/MyDrive/[컴비] 팀프로젝트/[공유폴더]"),
        "/content/drive/MyDrive/[공유폴더]",
        unicodedata.normalize("NFD", "/content/drive/MyDrive/[공유폴더]"),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    for root, dirs, _ in os.walk(base, followlinks=True):
        for d in dirs:
            if "공유폴더" in unicodedata.normalize("NFC", d):
                return os.path.join(root, d)
        if root.count(os.sep) - base.count(os.sep) >= 2:
            dirs[:] = []
    return candidates[0]

DATA_ROOT  = find_data_root()
print(f"📁 감지된 DATA_ROOT: {DATA_ROOT}")

REAL_DIR   = os.path.join(DATA_ROOT, "real")
FAKE_B_DIR = os.path.join(DATA_ROOT, "fake", "B")
MASK_DIR   = os.path.join(DATA_ROOT, "fake", "masks")
CSV_PATH   = os.path.join(DATA_ROOT, "fake", "fake_mapping_B.csv")

os.makedirs(FAKE_B_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

# 24종 공식 물건 목록
VALID_OBJECTS = [
    "smartphone", "laptop", "smartwatch", "earbuds", "headphones", "keyboard",
    "mouse", "sneakers", "tumbler", "hoodie", "chair", "wallet",
    "gamecontroller", "backpack", "book", "cosmetics", "fridge", "tripod",
    "lunchbox", "desk", "microphone", "muffler", "gloves", "perfume",
]

# 팀원 파일명 오타/단수형 자동 보정 매핑
OBJ_ALIAS = {
    "watch": "smartwatch",
    "headphone": "headphones",
    "headphonse": "headphones",
    "shose": "sneakers",
    "shoes": "sneakers",
    "shoe": "sneakers",
    "sneaker": "sneakers",
    "earbud": "earbuds",
    "glove": "gloves",
    "cosmetic": "cosmetics",
}

MEMO_B_COUNT_PER_OBJ   = 10
NOMEMO_B_COUNT_PER_OBJ = 5

def parse_real_filename(fname):
    stem = os.path.splitext(fname)[0]
    parts = [p.lower() for p in stem.split("_")]
    if len(parts) < 3:
        return None, None, False
    # nomemo / nememo 오타 & 위치(물건 앞/뒤) 모두 자동 인식
    is_nomemo = any(p in ("nomemo", "nememo") for p in parts[1:])
    if parts[1] in ("nomemo", "nememo") and len(parts) >= 4:
        raw_obj = parts[2]
    else:
        raw_obj = parts[1]
    obj_key = OBJ_ALIAS.get(raw_obj, raw_obj)
    if obj_key not in VALID_OBJECTS:
        # 오타 자동 보정 (유사도 70% 이상)
        matches = difflib.get_close_matches(obj_key, VALID_OBJECTS, n=1, cutoff=0.7)
        if matches:
            obj_key = matches[0]
    return obj_key, stem, is_nomemo

def scan_b_targets(real_dir=REAL_DIR):
    if not os.path.exists(real_dir):
        print(f"❌ 경로가 존재하지 않습니다: {real_dir}")
        print("📂 MyDrive 최상위 폴더 목록:", os.listdir("/content/drive/MyDrive"))
        return []

    # os.walk 사용: 경로에 [컴비], [공유폴더] 같은 대괄호가 있어도 100% 정상 탐색!
    all_imgs = []
    for root, _, files in os.walk(real_dir):
        for fn in files:
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                all_imgs.append(os.path.join(root, fn))
    all_imgs.sort()

    by_obj_memo   = defaultdict(list)
    by_obj_nomemo = defaultdict(list)
    unknown = []

    for p in all_imgs:
        fname = os.path.basename(p)
        obj_key, stem, is_nomemo = parse_real_filename(fname)
        if obj_key not in VALID_OBJECTS:
            unknown.append(fname)
            continue
        if is_nomemo:
            by_obj_nomemo[obj_key].append(p)
        else:
            by_obj_memo[obj_key].append(p)

    targets = []
    print(f"📂 전체 원본 이미지: {len(all_imgs)}장 (규약 외/미인식: {len(unknown)}장)")
    print("-" * 68)
    print(f"{'물건':<16} | {'원본(메모)':>8} | {'원본(노메모)':>10} | {'B선별(메모+노메모)':>18}")
    print("-" * 68)

    total_b_memo, total_b_nomemo = 0, 0
    for obj in VALID_OBJECTS:
        m_list = sorted(by_obj_memo[obj])
        nm_list = sorted(by_obj_nomemo[obj])

        # nomemo 태그 없이 001~030으로 올린 경우(예: 재환님): 뒤 5장(026~030)을 nomemo로 자동 분리
        if len(nm_list) == 0 and len(m_list) >= 26:
            nm_list = m_list[-NOMEMO_B_COUNT_PER_OBJ:]
            m_list  = m_list[:-NOMEMO_B_COUNT_PER_OBJ]

        sel_m  = m_list[-MEMO_B_COUNT_PER_OBJ:] if len(m_list) >= MEMO_B_COUNT_PER_OBJ else m_list
        sel_nm = nm_list[:NOMEMO_B_COUNT_PER_OBJ]
        total_b_memo   += len(sel_m)
        total_b_nomemo += len(sel_nm)
        for p in sel_m:
            targets.append((p, obj, False))
        for p in sel_nm:
            targets.append((p, obj, True))
        print(f"{obj:<16} | {len(m_list):>8} | {len(nm_list):>10} | {len(sel_m):>4} + {len(sel_nm):<4} = {len(sel_m)+len(sel_nm):>2}장")

    print("-" * 68)
    print(f"🎯 가짜 B 최종 생성 대상: 총 {len(targets)}장 (메모 {total_b_memo} + 노메모 {total_b_nomemo})")
    if unknown:
        print(f"⚠️ 규약 외 파일 ({len(unknown)}개): {unknown[:10]}")
    return targets

B_TARGETS = scan_b_targets()

## 3️⃣ [Cell 3] 모델 로드 (CLIPSeg + SDXL Inpainting)
> **이미지를 새로 생성할 때만** 실행하세요. (검수/원복만 할 때는 건너뛰기)


In [ ]:
# ==============================================================================
# 📌 [Cell 3] 모델 로드 (CLIPSeg 자동 마스크 + SDXL Inpainting 0.1)
# ==============================================================================
import torch
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from diffusers import AutoPipelineForInpainting, AutoencoderKL

INPAINT_MODEL_ID   = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
INPAINT_MODEL_NAME = "sdxl-inpainting-0.1"

print("1/2 CLIPSeg 로드 중...")
clipseg_processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
clipseg_model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined").to("cuda")
clipseg_model.eval()

print("2/2 SDXL Inpainting 로드 중...")
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16
)
pipe = AutoPipelineForInpainting.from_pretrained(
    INPAINT_MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")
pipe.vae.enable_slicing()

print("✅ CLIPSeg + SDXL Inpainting 로드 완료!")

## 4️⃣ [Cell 4] 핵심 생성 엔진 로드
> **이미지를 새로 생성할 때만** 실행하세요.


In [ ]:
# ==============================================================================
# 📌 [Cell 4] 핵심 엔진: 24종 다양화 프롬프트 + 메모지 보호 마스크 + 원본 페더 합성
# ==============================================================================
import csv, random, time
import numpy as np
from PIL import Image, ImageFilter, ImageOps

FAKE_MAPPING_COLUMNS = [
    "fake_filename", "fake_type", "source_filename", "mask_filename", "object",
    "prompt", "negative_prompt", "seed", "model", "created_by", "is_reinforcement",
]

# 물건별 (CLIPSeg 마스크 타깃, 다양한 교체 프롬프트 변형 리스트, threshold, strength)
# -> 66장 정밀 튜닝: strength 0.60~0.68로 낮춰 손가락/키캡/시계화면 찌그러짐 방지 + threshold 0.25~0.30으로 손목·다리 잘림 방지!
OBJ_CONFIG = {
    "smartphone":     ("smartphone", [
        "a realistic matte black smartphone lying flat, natural lighting, matching shadows, authentic photo",
        "a realistic silver smartphone with dark screen off, natural lighting, matching shadows, authentic photo",
        "a realistic navy blue smartphone in a slim silicone case, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "laptop":         ("laptop, laptop computer", [
        "a realistic silver aluminum laptop computer, clean screen and keyboard, natural lighting, matching shadows, authentic photo",
        "a realistic dark space gray laptop computer, clean matte finish, natural lighting, matching shadows, authentic photo",
        "a realistic matte black laptop computer with screen off, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.65),
    "smartwatch":     ("smartwatch, wristwatch, watch strap", [
        "a realistic matte black smartwatch with dark glass screen and black silicone band, same angle, natural lighting, matching shadows, authentic photo",
        "a realistic silver aluminum smartwatch with dark glass screen and gray band, same angle, natural lighting, matching shadows, authentic photo",
        "a realistic dark navy smartwatch with blank dark display and navy band, same angle, natural lighting, matching shadows, authentic photo",
    ], 0.24, 0.46),
    "earbuds":        ("wireless earbuds case, earphones", [
        "a realistic white wireless earbuds charging case, natural lighting, matching shadows, authentic photo",
        "a realistic matte black wireless earbuds charging case, natural lighting, matching shadows, authentic photo",
        "a realistic beige wireless earbuds charging case, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "headphones":     ("headphones", [
        "a realistic pair of matte black over-ear headphones, natural lighting, matching shadows, authentic photo",
        "a realistic pair of silver and gray over-ear headphones, natural lighting, matching shadows, authentic photo",
        "a realistic pair of beige wireless over-ear headphones, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "keyboard":       ("computer keyboard, keyboard", [
        "a realistic neat matte black mechanical computer keyboard with straight clean keys, same angle, natural lighting, matching shadows, authentic photo",
        "a realistic neat white and gray wireless computer keyboard with clean rectangular keys, same angle, natural lighting, matching shadows, authentic photo",
        "a realistic neat dark gray computer keyboard with uniform clean keycaps, same angle, natural lighting, matching shadows, authentic photo",
    ], 0.25, 0.48),
    "mouse":          ("computer mouse, wireless mouse", [
        "a realistic sleek matte black wireless computer mouse, smooth ergonomic shape, same angle, natural lighting, matching shadows, authentic photo",
        "a realistic sleek white wireless computer mouse, clean smooth surface, same angle, natural lighting, matching shadows, authentic photo",
        "a realistic sleek dark gray optical computer mouse, same angle, natural lighting, matching shadows, authentic photo",
    ], 0.25, 0.52),
    "sneakers":       ("sneakers, shoes", [
        "a realistic pair of white leather casual sneakers, natural lighting, matching shadows, authentic photo",
        "a realistic pair of gray running sneakers, natural lighting, matching shadows, authentic photo",
        "a realistic pair of black and white canvas sneakers, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "tumbler":        ("tumbler cup, thermos bottle, water bottle", [
        "a realistic matte black stainless steel insulated tumbler bottle, clean smooth body, natural lighting, matching shadows, authentic photo",
        "a realistic brushed silver metal insulated tumbler cup, natural lighting, matching shadows, authentic photo",
        "a realistic cream white insulated tumbler bottle, natural lighting, matching shadows, authentic photo",
    ], 0.28, 0.68),
    "hoodie":         ("hoodie, sweatshirt, clothing", [
        "a realistic folded heather gray cotton hoodie, natural fabric texture, natural lighting, matching shadows, authentic photo",
        "a realistic folded dark navy fleece hoodie, natural fabric folds, natural lighting, matching shadows, authentic photo",
        "a realistic folded charcoal black pullover hoodie, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "chair":          ("chair", [
        "a realistic black mesh ergonomic office chair, natural lighting, matching shadows, authentic photo",
        "a realistic gray fabric modern desk chair, natural lighting, matching shadows, authentic photo",
        "a realistic wooden dining chair with dark cushion, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "wallet":         ("leather wallet, wallet", [
        "a realistic dark brown leather bifold wallet, crisp edges, natural lighting, matching shadows, authentic photo",
        "a realistic matte black leather card wallet, clean stitching, natural lighting, matching shadows, authentic photo",
        "a realistic tan pebbled leather wallet, natural lighting, matching shadows, authentic photo",
    ], 0.28, 0.68),
    "gamecontroller": ("game controller, gamepad", [
        "a realistic matte black wireless game controller, natural lighting, matching shadows, authentic photo",
        "a realistic white and black wireless game controller, natural lighting, matching shadows, authentic photo",
        "a realistic dark blue wireless gamepad controller, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.65),
    "car":            ("car, automobile, vehicle", [
        "a realistic silver modern sedan car parked, natural lighting, matching shadows, authentic photo",
        "a realistic dark gray compact SUV car parked, natural lighting, matching shadows, authentic photo",
        "a realistic matte white hatchback car parked, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "backpack":       ("backpack, bag", [
        "a realistic black nylon everyday backpack, natural lighting, matching shadows, authentic photo",
        "a realistic dark gray canvas laptop backpack, natural lighting, matching shadows, authentic photo",
        "a realistic navy blue casual fabric backpack, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "book":           ("book", [
        "a realistic closed solid navy cloth hardcover book, completely blank plain cover with no text and no letters, natural lighting, matching shadows, authentic photo",
        "a realistic closed solid beige linen hardcover book, completely blank plain cover with no text and no letters, natural lighting, matching shadows, authentic photo",
        "a realistic closed solid dark green cloth hardcover book, completely blank plain cover with no text and no letters, natural lighting, matching shadows, authentic photo",
    ], 0.28, 0.82),
    "cosmetics":      ("cosmetics bottle, skincare bottle, cosmetic tube", [
        "a realistic plain white skincare lotion pump bottle, clean smooth surface, natural lighting, matching shadows, authentic photo",
        "a realistic frosted glass serum dropper bottle, clean silhouette, natural lighting, matching shadows, authentic photo",
        "a realistic plain beige cosmetic cream tube bottle, natural lighting, matching shadows, authentic photo",
    ], 0.28, 0.68),
    "fridge":         ("refrigerator, mini fridge", [
        "a realistic brushed silver mini refrigerator, clean flat door, natural lighting, matching shadows, authentic photo",
        "a realistic matte white compact refrigerator, clean edges, natural lighting, matching shadows, authentic photo",
        "a realistic charcoal gray modern refrigerator, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "tripod":         ("camera tripod, tripod stand", [
        "a realistic black aluminum camera tripod, straight clean legs, natural lighting, matching shadows, authentic photo",
        "a realistic dark gray carbon fiber camera tripod stand, straight legs, natural lighting, matching shadows, authentic photo",
        "a realistic compact black tabletop camera tripod, natural lighting, matching shadows, authentic photo",
    ], 0.25, 0.65),
    "lunchbox":       ("lunchbox container, food container", [
        "a realistic matte gray plastic bento lunchbox container, natural lighting, matching shadows, authentic photo",
        "a realistic clear and navy lid food lunchbox container, natural lighting, matching shadows, authentic photo",
        "a realistic beige two-tier bento lunchbox, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "desk":           ("desk, table", [
        "a realistic natural oak wooden study desk, natural lighting, matching shadows, authentic photo",
        "a realistic matte white modern computer desk, natural lighting, matching shadows, authentic photo",
        "a realistic dark walnut wooden desk surface, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "microphone":     ("microphone", [
        "a realistic matte black desktop USB microphone on a stand, natural lighting, matching shadows, authentic photo",
        "a realistic silver condenser microphone with desk stand, natural lighting, matching shadows, authentic photo",
        "a realistic dark gray podcast microphone with foam windscreen, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "muffler":        ("knitted scarf, muffler", [
        "a realistic folded charcoal gray knitted winter scarf, natural lighting, matching shadows, authentic photo",
        "a realistic folded camel beige wool scarf, natural lighting, matching shadows, authentic photo",
        "a realistic folded dark navy knitted muffler, natural lighting, matching shadows, authentic photo",
    ], 0.30, 0.68),
    "gloves":         ("gloves, winter gloves, fabric gloves", [
        "a realistic pair of black ribbed knit wool winter gloves, natural fabric texture, natural lighting, matching shadows, authentic photo",
        "a realistic pair of dark navy fleece winter gloves, clean stitched seams, natural lighting, matching shadows, authentic photo",
        "a realistic pair of camel brown suede winter gloves, natural fabric texture, natural lighting, matching shadows, authentic photo",
    ], 0.20, 0.42),
    "perfume":        ("perfume bottle, glass bottle", [
        "a realistic clear square glass perfume bottle with black cap, clean edges, natural lighting, matching shadows, authentic photo",
        "a realistic amber tinted glass perfume bottle with silver cap, clean silhouette, natural lighting, matching shadows, authentic photo",
        "a realistic minimalist cylindrical glass perfume bottle, natural lighting, matching shadows, authentic photo",
    ], 0.28, 0.68),
}

NEG_PROMPT_B = (
    "cartoon, 3d render, illustration, deformed, mutated, extra fingers, six fingers, "
    "distorted keys, warped screen, ghosting, double exposure, double object, blurry, "
    "low quality, watermark, logo, text, sticky note, memo paper, handwriting, paper note, "
    "artifacts, unnatural seam"
)

def dilate_mask_pil(mask_img: Image.Image, k: int) -> Image.Image:
    """PyTorch max_pool2d 기반 마스크 팽창 (Colab Pillow MaxFilter 버전 버그 우회)"""
    if k <= 1:
        return mask_img
    k = k if k % 2 == 1 else k + 1
    t = torch.from_numpy(np.array(mask_img, dtype=np.float32))[None, None]
    out = torch.nn.functional.max_pool2d(t, kernel_size=k, stride=1, padding=k // 2)
    return Image.fromarray(out[0, 0].clamp(0, 255).byte().numpy(), mode="L")

def make_protected_mask(
    pil_img: Image.Image,
    target_text: str,
    is_nomemo: bool = False,
    threshold: float = 0.30,
    dilate_px: int = 25,
) -> Image.Image:
    """CLIPSeg로 제품 영역만 추출하되, 메모 있는 사진에서는 메모지 영역을 강제 차감하여 보호"""
    texts = [target_text] if is_nomemo else [target_text, "sticky note, paper memo, handwritten note, white paper note, small piece of paper"]
    inputs = clipseg_processor(
        text=texts,
        images=[pil_img] * len(texts),
        padding="max_length",
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        logits = clipseg_model(**inputs).logits
    if logits.ndim == 2:
        logits = logits.unsqueeze(0)
    probs = torch.sigmoid(logits).cpu().numpy()

    obj_prob = probs[0]
    if not is_nomemo and len(probs) > 1:
        memo_prob = probs[1]
        final_prob = np.clip(obj_prob - memo_prob * 1.25, 0.0, 1.0)
    else:
        final_prob = obj_prob

    # 마스크 면적 안전장치 (너무 작거나 너무 크면 threshold 자동 보정)
    th = threshold
    mask_bin = (final_prob > th)
    ratio = mask_bin.mean()
    if ratio < 0.03:
        mask_bin = (final_prob > max(0.12, th - 0.12))
    elif ratio > 0.78:
        mask_bin = (final_prob > min(0.65, th + 0.15))

    mask_u8 = (mask_bin.astype(np.uint8) * 255)
    mask_img = Image.fromarray(mask_u8).resize(pil_img.size, Image.BILINEAR)
    mask_img = mask_img.point(lambda x: 255 if x >= 128 else 0)

    if dilate_px > 0:
        mask_img = dilate_mask_pil(mask_img, dilate_px)

    # dilate 이후에도 메모지 영역 침범 차단 (desk 등 큰 배경 물건에서도 메모지 확실히 보호)
    if not is_nomemo and len(probs) > 1:
        memo_u8 = ((probs[1] > 0.22).astype(np.uint8) * 255)
        memo_mask = Image.fromarray(memo_u8).resize(pil_img.size, Image.BILINEAR)
        memo_mask = dilate_mask_pil(memo_mask, 19)
        m_arr = np.array(mask_img)
        m_arr[np.array(memo_mask) >= 128] = 0
        mask_img = Image.fromarray(m_arr)

    return mask_img.convert("L")

def composite_with_feather(orig_img: Image.Image, gen_img: Image.Image, mask_img: Image.Image, feather_px: int = 4) -> Image.Image:
    """마스크 밖(배경·메모지)은 100% 원본 픽셀 유지 + 얇은 페더 블렌딩(원본 잔상/번짐 방지)"""
    gen_resized = gen_img.resize(orig_img.size, Image.LANCZOS)
    k = feather_px * 2 + 1
    t = torch.from_numpy(np.array(mask_img, dtype=np.float32))[None, None] / 255.0
    t = torch.nn.functional.avg_pool2d(t, kernel_size=k, stride=1, padding=feather_px)
    alpha = t[0, 0].clamp(0.0, 1.0).numpy()[..., None]
    blended = (np.array(gen_resized, dtype=np.float32) * alpha + np.array(orig_img, dtype=np.float32) * (1.0 - alpha))
    return Image.fromarray(blended.clip(0, 255).astype(np.uint8), mode="RGB")

def generate_fake_b(
    src_path: str,
    obj_key: str,
    is_nomemo: bool = False,
    seed: int = None,
    strength: float = None,
    guidance: float = 7.0,
    steps: int = 35,
    mask_threshold: float = None,
    created_by: str = "changjun",
    is_reinforcement: int = 0,
    record_log: bool = True,
):
    seed = seed if seed is not None else random.randint(1, 99999)
    rng = random.Random(seed)

    src_fname = os.path.basename(src_path)
    src_stem  = os.path.splitext(src_fname)[0]

    # EXIF 방향 보정 + SDXL 1024 기준 리사이즈 (8의 배수)
    orig = ImageOps.exif_transpose(Image.open(src_path)).convert("RGB")
    orig_w, orig_h = orig.size
    scale = 1024 / max(orig_w, orig_h)
    rw = max(512, int(round(orig_w * scale / 8) * 8))
    rh = max(512, int(round(orig_h * scale / 8) * 8))
    img_in = orig.resize((rw, rh), Image.LANCZOS)

    mask_target, prompt_list, default_th, default_str = OBJ_CONFIG.get(
        obj_key, (obj_key, [f"a realistic {obj_key}, natural lighting, matching shadows, authentic photo"], 0.30, 0.68)
    )
    prompt = rng.choice(prompt_list)
    th = mask_threshold if mask_threshold is not None else default_th
    eff_strength = strength if strength is not None else default_str

    eff_dilate = 21 if obj_key == "gloves" else 25
    mask_in = make_protected_mask(img_in, mask_target, is_nomemo=is_nomemo, threshold=th, dilate_px=eff_dilate)

    g = torch.Generator("cuda").manual_seed(seed)
    raw_out = pipe(
        prompt=prompt,
        negative_prompt=NEG_PROMPT_B,
        image=img_in,
        mask_image=mask_in,
        strength=eff_strength,
        guidance_scale=guidance,
        num_inference_steps=steps,
        generator=g,
    ).images[0]

    # 배경·메모지 원본 픽셀 100% 보존 합성 (얇은 4px 페더로 잔상 제거)
    final_img = composite_with_feather(img_in, raw_out, mask_in, feather_px=4)

    fake_stem  = f"fakeB_{src_stem}_s{seed}"
    fake_fname = f"{fake_stem}.png"
    mask_fname = f"{fake_stem}_mask.png"

    if record_log:
        final_img.save(os.path.join(FAKE_B_DIR, fake_fname))
        mask_in.save(os.path.join(MASK_DIR, mask_fname))

        new_file = not os.path.exists(CSV_PATH)
        with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new_file:
                w.writerow(FAKE_MAPPING_COLUMNS)
            w.writerow([
                fake_fname, "B", src_fname, mask_fname, obj_key,
                prompt, NEG_PROMPT_B, seed, INPAINT_MODEL_NAME, created_by, is_reinforcement,
            ])

    return img_in, mask_in, final_img, fake_fname

print("✅ 가짜 B 생성 엔진 로드 완료!")

## 🧪 [Cell 5] (선택) 파일럿 테스트 — 6종 눈 검수
> 이미 본 양산/재뽑기 단계라면 건너뛰세요.


In [ ]:
# ==============================================================================
# 📌 [Cell 5] Step 2: 파일럿 테스트 (물건당 1장씩 3단 그리드 눈 검수)
# ==============================================================================
import matplotlib.pyplot as plt

def run_pilot(target_objects=None, max_count=24):
    """물건당 1장씩만 뽑아서 [원본 | 마스크 | 가짜B] 3단 비교 (저장/CSV 기록 안 함)"""
    seen = set()
    samples = []
    for p, obj, is_nm in B_TARGETS:
        if target_objects and obj not in target_objects:
            continue
        if obj not in seen:
            seen.add(obj)
            samples.append((p, obj, is_nm))
        if len(samples) >= max_count:
            break

    print(f"🔍 파일럿 테스트 시작: {len(samples)}개 물건 (CSV 미기록)")
    for idx, (p, obj, is_nm) in enumerate(samples, 1):
        orig_im, mask_im, fake_im, fn = generate_fake_b(p, obj, is_nomemo=is_nm, record_log=False)
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(orig_im); axes[0].set_title(f"[{idx}] {obj} Orig"); axes[0].axis("off")
        axes[1].imshow(mask_im, cmap="gray"); axes[1].set_title("Protected Mask"); axes[1].axis("off")
        axes[2].imshow(fake_im); axes[2].set_title("Fake B Result"); axes[2].axis("off")
        plt.tight_layout()
        plt.show()

# 👉 처음엔 6종만 빠르게 보거나, 바로 전체 24종 파일럿 실행:
run_pilot(max_count=6)
# run_pilot(max_count=24)

## 🚀 [Cell 6] 본 양산 / 부족분 자동 재뽑기 (`run_batch`)
> 이미 `fake/B/`에 있는 통과본은 자동 스킵하고 **부족한 장수만** 새로 생성합니다.


In [ ]:
# ==============================================================================
# 📌 [Cell 6] Step 3: 360장 본 양산 (실시간 드라이브 저장 + 자동 이어받기)
# ==============================================================================
def run_batch(target_objects=None):
    """
    B_TARGETS 전체(360장) 일괄 생성.
    이미 data/fake/B/에 생성된 원본은 자동 스킵(Resume)하므로 끊겨도 다시 누르면 됩니다!
    """
    queue = [
        (p, obj, is_nm) for (p, obj, is_nm) in B_TARGETS
        if (target_objects is None or obj in target_objects)
    ]
    total = len(queue)
    done_cnt, skip_cnt, err_cnt = 0, 0, 0
    t0 = time.time()

    print(f"🚀 가짜 B 본 양산 시작! 대상: {total}장")
    for i, (src_p, obj_key, is_nm) in enumerate(queue, 1):
        src_fname = os.path.basename(src_p)
        src_stem  = os.path.splitext(src_fname)[0]

        # 이미 생성된 원본이면 자동 스킵 (이어받기, 대괄호 경로 안전하게 os.listdir 사용)
        existing_b = os.listdir(FAKE_B_DIR) if os.path.exists(FAKE_B_DIR) else []
        if any(f.startswith(f"fakeB_{src_stem}_s") and f.endswith(".png") for f in existing_b):
            skip_cnt += 1
            continue

        try:
            _, _, _, fake_fn = generate_fake_b(src_p, obj_key, is_nomemo=is_nm, record_log=True)
            done_cnt += 1
            elapsed = time.time() - t0
            avg_sec = elapsed / max(1, done_cnt)
            remain  = (total - i) * avg_sec / 60.0
            print(f"[{i}/{total}] ✅ {src_fname} -> {fake_fn} (남은 예상: {remain:.1f}분)")
        except Exception as e:
            err_cnt += 1
            print(f"[{i}/{total}] ❌ 에러 ({src_fname}): {e}")

    print("=" * 60)
    print(f"🎉 배치 종료! 신규 생성: {done_cnt}장 | 스킵(기존완료): {skip_cnt}장 | 에러: {err_cnt}장")

# 👉 파일럿 확인 끝나면 아래 주석 해제 후 실행!
run_batch()
# 특정 물건(예: val/test 물건)부터 먼저 뽑고 싶으면:
# run_batch(target_objects={"mouse", "keyboard", "smartphone"})

---
# 🔍 검수 & 재뽑기 도우미 (아래 순서대로 실행)
> ⚠️ **주의**: `Cell 9 (sync_and_verify)`는 반드시 `Cell 8 (원복)`을 먼저 실행해서 사진이 `fake/B/`에 다 나와있는 상태에서 돌려야 합니다!


## 📂 [Cell 7] 검수 전: 새로 뽑힌 사진만 `_재검수/` 물건별 폴더로 모으기
> `Cell 6` 생성이 끝난 뒤 실행하세요. 이미 통과된 사진은 건드리지 않고 새로 뽑힌 사진만 모읍니다.


In [ ]:
# ==============================================================================
# 📌 [Cell 7] 검수 전: 새로 뽑힌 사진만 fake/B/_재검수/ (물건별 폴더)로 모으기
# ==============================================================================
import os, shutil
import pandas as pd

APPROVED   = os.path.join(DATA_ROOT, "fake", "_approved_B.txt")
REVIEW_DIR = os.path.join(FAKE_B_DIR, "_재검수")

if os.path.exists(APPROVED):
    with open(APPROVED, encoding="utf-8") as f:
        done = set(line.strip() for line in f if line.strip())
else:
    done = set()

df = pd.read_csv(CSV_PATH).drop_duplicates(subset=["fake_filename"], keep="last")
obj_map = dict(zip(df["fake_filename"], df["object"]))

new_cnt = 0
for fn in sorted(os.listdir(FAKE_B_DIR)):
    if fn.startswith("fakeB_") and fn.endswith(".png") and fn not in done:
        sub = os.path.join(REVIEW_DIR, obj_map.get(fn, "etc"))
        os.makedirs(sub, exist_ok=True)
        shutil.move(os.path.join(FAKE_B_DIR, fn), os.path.join(sub, fn))
        new_cnt += 1

print(f"🔍 새로 뽑힌 {new_cnt}장을 fake/B/_재검수/ (물건별 폴더)로 모았습니다!")
print("👉 이제 구글 드라이브 fake/B/_재검수/ 에서 망한 사진만 삭제한 뒤 Cell 8을 실행하세요.")

## 📦 [Cell 8] 검수 후 ①: 폴더에서 사진 전부 원복 + 빈 폴더 삭제
> 드라이브 `fake/B/_재검수/`에서 망한 사진을 다 지운 뒤 실행하세요.


In [ ]:
# ==============================================================================
# 📌 [Cell 8] 검수 후 ①: 하위 폴더에서 사진 전부 fake/B/로 원복 + 빈 폴더 삭제
# ==============================================================================
import os, shutil

restored = 0
for root, dirs, files in os.walk(FAKE_B_DIR, topdown=False):
    if root == FAKE_B_DIR:
        continue
    for fn in files:
        shutil.move(os.path.join(root, fn), os.path.join(FAKE_B_DIR, fn))
        restored += 1
    if len(os.listdir(root)) == 0:
        os.rmdir(root)

print(f"✅ {restored}장 원복 완료! 이제 바로 아래 Cell 9 (sync_and_verify)를 실행하세요.")

## 🧹 [Cell 9] 검수 후 ②: 마스크·CSV 1:1 정리 + 부족분 리포트 (`sync_and_verify`)
> 반드시 **Cell 8 원복 직후** 실행하세요!


In [ ]:
# ==============================================================================
# 📌 [Cell 7] Step 4~5: 검수 후 자동 정리 & 1:1 정합성 검증
# ==============================================================================
# 사용법: 구글 드라이브 data/fake/B/ 에서 망한 이미지만 삭제한 뒤 이 셀을 실행하세요!
# -> 짝꿍 마스크 자동 삭제 + fake_mapping_B.csv 자동 동기화 + 부족분 리포트 출력

import pandas as pd
from collections import Counter

def sync_and_verify():
    b_files = sorted(
        f for f in os.listdir(FAKE_B_DIR)
        if f.startswith("fakeB_") and f.endswith(".png")
    )
    b_set = set(b_files)
    valid_masks = {os.path.splitext(f)[0] + "_mask.png" for f in b_files}

    # 1) 고아 마스크(이미지가 삭제된 마스크) 자동 삭제
    all_b_masks = [
        os.path.join(MASK_DIR, f) for f in os.listdir(MASK_DIR)
        if f.startswith("fakeB_") and f.endswith("_mask.png")
    ]
    removed_masks = 0
    for mp in all_b_masks:
        if os.path.basename(mp) not in valid_masks:
            os.remove(mp)
            removed_masks += 1

    # 2) CSV 동기화 (삭제된 행 제거 + 중복 제거)
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        before_rows = len(df)
        df = df[df["fake_filename"].isin(b_set)].drop_duplicates(subset=["fake_filename"], keep="last")
        df.to_csv(CSV_PATH, index=False, encoding="utf-8")
        after_rows = len(df)
    else:
        before_rows = after_rows = 0
        df = pd.DataFrame(columns=FAKE_MAPPING_COLUMNS)

    # 3) 1:1 정합성 확인
    cur_masks = [
        f for f in os.listdir(MASK_DIR)
        if f.startswith("fakeB_") and f.endswith("_mask.png")
    ]
    print(f"🧹 고아 마스크 삭제: {removed_masks}개 | CSV 정리: {before_rows}행 -> {after_rows}행")
    print(f"📊 최종 현황: B 이미지 {len(b_files)}장 | B 마스크 {len(cur_masks)}장 | CSV {after_rows}행")

    if len(b_files) == len(cur_masks) == after_rows:
        print("✅ 이미지 == 마스크 == CSV 1:1 완전 일치!")
    else:
        print("⚠️ 장수 불일치! 누락된 마스크나 CSV 행을 확인하세요.")

    # 4) 물건별 부족분 리포트 (15장 기준)
    if len(df) > 0:
        cnt = Counter(df["object"])
        lacking = [(obj, cnt.get(obj, 0)) for obj in VALID_OBJECTS if cnt.get(obj, 0) < 15]
        if lacking:
            print(f"📌 15장 미만 물건 ({len(lacking)}종) -> Cell 6 run_batch() 다시 실행하면 자동 보충:")
            for obj, c in lacking:
                print(f"   - {obj}: {c}/15장 (부족 {15 - c}장)")
        else:
            print("🏆 24종 전 물건 15장씩 총 360장 100% 완성!!")

sync_and_verify()

## 💾 [Cell 10] 재뽑기 직전: 현재 통과본 목록 저장 (`_approved_B.txt`)
> Cell 9에서 부족분이 남아있으면 이 셀을 실행해서 통과본을 기록한 뒤, **Cell 6**으로 돌아가세요!


In [ ]:
# ==============================================================================
# 📌 [Cell 10] 재뽑기 직전: 현재 통과본 목록 저장 (_approved_B.txt)
# ==============================================================================
# Cell 9에서 부족분이 있을 때, Cell 6(run_batch) 돌리기 직전에 실행하세요!
# (360장 100% 완성되면 아래 맨 끝 줄 주석 풀어서 임시 파일 _approved_B.txt 삭제)
import os

APPROVED = os.path.join(DATA_ROOT, "fake", "_approved_B.txt")
done = sorted(f for f in os.listdir(FAKE_B_DIR) if f.startswith("fakeB_") and f.endswith(".png"))
with open(APPROVED, "w", encoding="utf-8") as f:
    f.write("\n".join(done))

print(f"✅ 현재 통과본 {len(done)}장 기록 완료! 이제 GPU 켜고 Cell 6 (run_batch) 실행하세요.")

## 🏆 [Cell 11] 360장 최종 완료 마감 체크 & 임시파일 자동 정리
> **360장 100% 완성된 뒤 맨 마지막에 딱 1번만** 실행하세요! (360장·마스크·CSV 1:1 검증 + 깨진 이미지 검사 + `_approved_B.txt` 자동 삭제)


In [ ]:
# ==============================================================================
# 📌 [Cell 11] 최종 마감 검증 (360장 무결성 체크 + _approved_B.txt 자동 삭제)
# ==============================================================================
import os
import pandas as pd
from PIL import Image
from collections import Counter

b_files = sorted(f for f in os.listdir(FAKE_B_DIR) if f.startswith("fakeB_") and f.endswith(".png"))
m_files = sorted(f for f in os.listdir(MASK_DIR) if f.startswith("fakeB_") and f.endswith("_mask.png"))
subdirs = [d for d in os.listdir(FAKE_B_DIR) if os.path.isdir(os.path.join(FAKE_B_DIR, d))]
df = pd.read_csv(CSV_PATH) if os.path.exists(CSV_PATH) else pd.DataFrame()

# 1) 깨진 이미지 / 0바이트 파일 검사
broken = []
for fn in b_files:
    p = os.path.join(FAKE_B_DIR, fn)
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        broken.append(fn)

# 2) 24종 × 15장 확인
cnt = Counter(df["object"]) if len(df) > 0 else Counter()
lacking = [(obj, cnt.get(obj, 0)) for obj in VALID_OBJECTS if cnt.get(obj, 0) != 15]

print(f"📊 최종 집계: 이미지 {len(b_files)}장 | 마스크 {len(m_files)}장 | CSV {len(df)}행")
print(f"🔍 깨진 이미지: {len(broken)}개 | 남은 하위폴더: {subdirs}")

all_ok = (
    len(b_files) == 360
    and len(m_files) == 360
    and len(df) == 360
    and len(broken) == 0
    and len(subdirs) == 0
    and len(lacking) == 0
)

if all_ok:
    APPROVED = os.path.join(DATA_ROOT, "fake", "_approved_B.txt")
    if os.path.exists(APPROVED):
        os.remove(APPROVED)
        print("🧹 임시 파일 _approved_B.txt 자동 삭제 완료!")
    print("=" * 60)
    print("🏆 가짜 B 360장 (24종 × 15장) 무결성 100% 통과!! 최종 납품 준비 완료!")
    print(f"   - 이미지: {FAKE_B_DIR}")
    print(f"   - 마스크: {MASK_DIR}")
    print(f"   - 대응표: {CSV_PATH}")
else:
    print("⚠️ 아직 360장 최종 조건을 충족하지 않아 _approved_B.txt를 유지합니다.")
    if broken:
        print(f"   - 깨진 파일: {broken[:10]}")
    if lacking:
        print(f"   - 15장 불일치 물건: {lacking}")